# Phase 4 — Silver Layer (Cleansing, Delta MERGE, SCD, Surrogate Keys)
**Apex Retail Intelligence | Celebal Technologies CEI'26 Major Project**

This is the most critical phase of the pipeline. It:
1. Applies Data Quality (DQ) rules to Bronze data (drop null PKs, dedup, type casting, null-fill).
2. Uses **Delta Lake `MERGE INTO`** (no watermarking) to reconcile historical + incremental batches.
3. Implements **SCD Type 2** for customers, **SCD Type 1** for products, and an **immutable,
   deduplicated ledger** for sales.
4. Generates surrogate keys (`customer_sk`, `product_sk`, `sales_sk`) for Gold-layer joins.

In [0]:
dbutils.widgets.text("bronze_catalog", "apex_retail1", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze_tables", "Bronze Schema")
dbutils.widgets.text("silver_schema", "silver_tables", "Silver Schema")
dbutils.widgets.text("catalog_root", "/Volumes/apex_retail1/landing_zone/inbound_data", "Audit Volume Root")

CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
CATALOG_ROOT = dbutils.widgets.get("catalog_root")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
import datetime as dt

TODAY = dt.date.today().isoformat()

## 4.1 Data Quality Rules & Cleansing
Applied uniformly to historical **and** incremental batches before any MERGE happens:
- Drop rows missing their primary key (`customer_id` / `product_id` / `transaction_id`).
- Remove duplicate records.
- Cast price/quantity fields to strict numeric types.
- Fill missing strings with `"Unknown"`, missing numerics with `0.0`.

In [0]:
def apply_dq_rules(df, pk_cols, numeric_cols):
    """Standard DQ pass: PK not-null, dedup, numeric casting, null-fill."""
    for pk in pk_cols:
        df = df.filter(F.col(pk).isNotNull() & (F.trim(F.col(pk)) != ""))
    df = df.dropDuplicates()
    for c in numeric_cols:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast("double"))
    string_cols = [c for c in df.columns if c not in numeric_cols and c not in pk_cols]
    for c in string_cols:
        df = df.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), F.lit("Unknown")).otherwise(F.col(c)))
    for c in numeric_cols:
        if c in df.columns:
            df = df.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    return df

def bronze(entity, batch=None):
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{entity}")
    if batch:
        df = df.filter(F.col("source_batch") == batch)
    return df.drop("ingested_at", "source_batch")

In [0]:
CUSTOMER_NUM = ["age", "membership_years", "number_of_children"]
PRODUCT_NUM = ["product_rating", "product_review_count", "product_stock", "product_return_rate",
               "product_weight", "product_shelf_life", "unit_price"]
SALES_NUM = ["quantity", "unit_price", "discount_applied", "total_sales"]

cust_hist_clean = apply_dq_rules(bronze("customer", "historical"), ["customer_id"], CUSTOMER_NUM)
cust_inc_clean = apply_dq_rules(bronze("customer", "incremental"), ["customer_id"], CUSTOMER_NUM)

prod_hist_clean = apply_dq_rules(bronze("product", "historical"), ["product_id"], PRODUCT_NUM)
prod_inc_clean = apply_dq_rules(bronze("product", "incremental"), ["product_id"], PRODUCT_NUM)

sales_hist_clean = apply_dq_rules(bronze("sales", "historical"), ["transaction_id"], SALES_NUM)
sales_inc_clean = apply_dq_rules(bronze("sales", "incremental"), ["transaction_id"], SALES_NUM)

print("Post-DQ row counts:")
for name, d in [("customer_historical", cust_hist_clean), ("customer_incremental", cust_inc_clean),
                 ("product_historical", prod_hist_clean), ("product_incremental", prod_inc_clean),
                 ("sales_historical", sales_hist_clean), ("sales_incremental", sales_inc_clean)]:
    print(f"  {name:24s} {d.count()}")

Post-DQ row counts:
  customer_historical      1051
  customer_incremental     1053
  product_historical       1042
  product_incremental      1041
  sales_historical         1000
  sales_incremental        1000


## 4.2 / 4.3 — Customer SCD Type 2 via Delta MERGE
Strategy: maintain `dim_customer_silver` with `effective_start_date`, `effective_end_date`,
`is_current`. On MERGE:
- If an incoming row's key exists **and any tracked attribute changed** → expire the current
  row (`is_current = false`, `effective_end_date = today`) and insert a new active version.
- If the key is new → insert as a new active row.
- Two-pass MERGE pattern (expire pass + insert pass), the standard Delta SCD2 recipe since a
  single `MERGE` can't both update-old-row and insert-new-row for the same source record.

In [0]:
# ============================================================
# COMPLETE CUSTOMER SCD TYPE 2 PIPELINE - SINGLE CELL
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

customer_table = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer"

CUSTOMER_TRACKED_COLS = [
    c for c in cust_hist_clean.columns
    if c != "customer_id"
]

print("=" * 70)
print("CUSTOMER SCD TYPE 2 PIPELINE")
print("=" * 70)
print(f"Target table: {customer_table}")


# ============================================================
# STEP 1: CREATE / SEED DIM_CUSTOMER
# ============================================================

if not spark.catalog.tableExists(customer_table):

    print("\nCreating initial SCD2 dimension...")

    seed_window = Window.orderBy("customer_id")

    seed = (
        cust_hist_clean
        .dropDuplicates(["customer_id"])
        .withColumn(
            "effective_start_date",
            F.to_date(F.lit("2020-01-01"))
        )
        .withColumn(
            "effective_end_date",
            F.lit(None).cast("date")
        )
        .withColumn(
            "is_current",
            F.lit(True)
        )
        .withColumn(
            "customer_sk",
            F.row_number().over(seed_window)
        )
    )

    seed.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(customer_table)

    print(
        f"Seeded {customer_table} with "
        f"{seed.count()} historical rows."
    )

else:

    print("\nExisting dim_customer found. No reseeding required.")


# ============================================================
# STEP 2: LOAD TARGET DIMENSION
# ============================================================

dim_customer = DeltaTable.forName(
    spark,
    customer_table
)

target_df = dim_customer.toDF()


# ============================================================
# STEP 3: REPAIR EXISTING DUPLICATE ACTIVE RECORDS
# ============================================================

duplicate_active = (
    target_df
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_active.count()

if duplicate_count > 0:

    print(
        f"\nWARNING: Found {duplicate_count} customer IDs "
        "with multiple active records."
    )

    duplicate_active.show()

    # Keep the latest active record.
    # Expire all older active duplicates.
    repair_window = (
        Window
        .partitionBy("customer_id")
        .orderBy(
            F.col("effective_start_date").desc(),
            F.col("customer_sk").desc()
        )
    )

    duplicate_rows_to_expire = (
        target_df
        .filter(F.col("is_current") == True)
        .withColumn(
            "_repair_rank",
            F.row_number().over(repair_window)
        )
        .filter(F.col("_repair_rank") > 1)
        .select("customer_sk")
    )

    repair_count = duplicate_rows_to_expire.count()

    if repair_count > 0:

        repair_ids = [
            row["customer_sk"]
            for row in duplicate_rows_to_expire.collect()
        ]

        id_list = ",".join(
            [str(int(x)) for x in repair_ids]
        )

        spark.sql(f"""
            UPDATE {customer_table}
            SET
                is_current = false,
                effective_end_date = TO_DATE('{TODAY}')
            WHERE customer_sk IN ({id_list})
        """)

        print(
            f"Repaired {repair_count} duplicate active records."
        )

else:

    print("\nNo duplicate active records found.")


# ============================================================
# STEP 4: RELOAD TARGET AFTER REPAIR
# ============================================================

dim_customer = DeltaTable.forName(
    spark,
    customer_table
)

target_df = dim_customer.toDF()

current_view = (
    target_df
    .filter(F.col("is_current") == True)
)


# ============================================================
# STEP 5: VALIDATE TARGET
# ============================================================

remaining_duplicates = (
    current_view
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

remaining_duplicate_count = remaining_duplicates.count()

if remaining_duplicate_count > 0:

    print(
        "\nERROR: Duplicate active records still exist:"
    )

    remaining_duplicates.show()

    raise ValueError(
        "Cannot continue SCD2 because duplicate active "
        "customer IDs still exist."
    )

print("Current dim_customer validation passed.")


# ============================================================
# STEP 6: PREPARE HISTORICAL DATA
# ============================================================

hist = (
    cust_hist_clean
    .withColumn(
        "_source_priority",
        F.lit(1)
    )
)


# ============================================================
# STEP 7: PREPARE INCREMENTAL DATA
# ============================================================

inc = (
    cust_inc_clean
    .withColumn(
        "_source_priority",
        F.lit(2)
    )
)


# ============================================================
# STEP 8: COMBINE HISTORICAL + INCREMENTAL
# ============================================================

incoming_raw = hist.unionByName(inc)


# ============================================================
# STEP 9: DEDUPLICATE INCOMING DATA
# Incremental record wins if customer exists in both.
# ============================================================

incoming_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_source_priority").desc()
    )
)

incoming_customers = (
    incoming_raw
    .withColumn(
        "_rn",
        F.row_number().over(incoming_window)
    )
    .filter(F.col("_rn") == 1)
    .drop(
        "_rn",
        "_source_priority"
    )
)


# ============================================================
# STEP 10: VALIDATE INCOMING DATA
# ============================================================

incoming_duplicates = (
    incoming_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

incoming_duplicate_count = incoming_duplicates.count()

if incoming_duplicate_count > 0:

    incoming_duplicates.show()

    raise ValueError(
        f"Incoming source contains "
        f"{incoming_duplicate_count} duplicate customer IDs."
    )

incoming_count = (
    incoming_customers
    .select("customer_id")
    .distinct()
    .count()
)

print(
    f"Incoming customer validation passed: "
    f"{incoming_count} unique customers."
)


# ============================================================
# STEP 11: NULL-SAFE CHANGE DETECTION
# ============================================================

change_conditions = []

for c in CUSTOMER_TRACKED_COLS:

    change_conditions.append(
        f"NOT (tgt.`{c}` <=> src.`{c}`)"
    )

change_cond = " OR ".join(change_conditions)


# ============================================================
# STEP 12: FIND CHANGED CUSTOMERS
# ============================================================

changed = (
    incoming_customers.alias("src")
    .join(
        current_view.alias("tgt"),
        F.col("src.customer_id") ==
        F.col("tgt.customer_id"),
        "inner"
    )
    .where(F.expr(change_cond))
    .select("src.*")
    .dropDuplicates(["customer_id"])
)


# ============================================================
# STEP 13: FIND BRAND-NEW CUSTOMERS
# ============================================================

new_keys = (
    incoming_customers
    .join(
        current_view.select("customer_id"),
        "customer_id",
        "left_anti"
    )
    .dropDuplicates(["customer_id"])
)


n_changed = (
    changed
    .select("customer_id")
    .distinct()
    .count()
)

n_new = (
    new_keys
    .select("customer_id")
    .distinct()
    .count()
)

print(f"Changed customers detected: {n_changed}")
print(f"New customers detected: {n_new}")


# ============================================================
# STEP 14: VALIDATE CHANGED SOURCE
# ============================================================

changed_duplicates = (
    changed
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

changed_duplicate_count = changed_duplicates.count()

if changed_duplicate_count > 0:

    changed_duplicates.show()

    raise ValueError(
        "SCD2 MERGE aborted because changed source "
        "contains duplicate customer IDs."
    )


# ============================================================
# STEP 15: EXPIRE OLD VERSIONS
# ============================================================

if n_changed > 0:

    (
        dim_customer.alias("tgt")
        .merge(
            changed.alias("src"),
            """
            tgt.customer_id = src.customer_id
            AND tgt.is_current = true
            """
        )
        .whenMatchedUpdate(
            set={
                "is_current": "false",
                "effective_end_date":
                    f"TO_DATE('{TODAY}')"
            }
        )
        .execute()
    )

    print(
        f"Expired {n_changed} old customer versions."
    )

else:

    print("No existing customers changed.")


# ============================================================
# STEP 16: PREPARE NEW ACTIVE VERSIONS
# ============================================================

to_insert = (
    changed
    .unionByName(new_keys)
    .dropDuplicates(["customer_id"])
)


# ============================================================
# STEP 17: GENERATE SURROGATE KEYS
# ============================================================

max_sk = (
    dim_customer
    .toDF()
    .agg(F.max("customer_sk"))
    .collect()[0][0]
)

if max_sk is None:
    max_sk = 0


insert_window = Window.orderBy("customer_id")


to_insert_final = (
    to_insert
    .withColumn(
        "effective_start_date",
        F.to_date(F.lit(TODAY))
    )
    .withColumn(
        "effective_end_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        F.lit(True)
    )
    .withColumn(
        "row_num",
        F.row_number().over(insert_window)
    )
    .withColumn(
        "customer_sk",
        F.col("row_num") + F.lit(max_sk)
    )
    .drop("row_num")
)


# ============================================================
# STEP 18: INSERT NEW ACTIVE VERSIONS
# ============================================================

insert_count = (
    to_insert_final
    .select("customer_id")
    .distinct()
    .count()
)

if insert_count > 0:

    (
        dim_customer.alias("tgt")
        .merge(
            to_insert_final.alias("src"),
            """
            tgt.customer_id = src.customer_id
            AND tgt.is_current = true
            AND 1 = 0
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Inserted {insert_count} new active customer versions."
    )

else:

    print("No new customer versions to insert.")


# ============================================================
# STEP 19: FINAL VALIDATION
# ============================================================

final_df = spark.table(customer_table)


final_active_duplicates = (
    final_df
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

final_duplicate_count = final_active_duplicates.count()


n_total = final_df.count()

n_active = (
    final_df
    .filter(F.col("is_current") == True)
    .count()
)

n_history = (
    final_df
    .filter(F.col("is_current") == False)
    .count()
)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("SCD TYPE 2 CUSTOMER PIPELINE COMPLETE")
print("=" * 70)

print(f"Target table          : {customer_table}")
print(f"Incoming customers    : {incoming_count}")
print(f"Changed customers     : {n_changed}")
print(f"New customers         : {n_new}")
print(f"Total dimension rows  : {n_total}")
print(f"Active rows           : {n_active}")
print(f"Historical rows       : {n_history}")
print(f"Duplicate active IDs  : {final_duplicate_count}")

print("=" * 70)


if final_duplicate_count == 0:

    print("SUCCESS: SCD2 validation passed.")
    print("There are no duplicate active customer IDs.")

else:

    print(
        "ERROR: Duplicate active customer IDs remain."
    )

    final_active_duplicates.show()

    raise ValueError(
        "Final SCD2 validation failed."
    )

CUSTOMER SCD TYPE 2 PIPELINE
Target table: apex_retail1.silver_tables.dim_customer

Existing dim_customer found. No reseeding required.

+-----------+-----+
|customer_id|count|
+-----------+-----+
|          4|    2|
+-----------+-----+

Repaired 1 duplicate active records.
Current dim_customer validation passed.
Incoming customer validation passed: 1050 unique customers.
Changed customers detected: 1050
New customers detected: 0
Expired 1050 old customer versions.


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Inserted 1050 new active customer versions.

SCD TYPE 2 CUSTOMER PIPELINE COMPLETE
Target table          : apex_retail1.silver_tables.dim_customer
Incoming customers    : 1050
Changed customers     : 1050
New customers         : 0
Total dimension rows  : 2101
Active rows           : 1050
Historical rows       : 1051
Duplicate active IDs  : 0
SUCCESS: SCD2 validation passed.
There are no duplicate active customer IDs.


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 4.3 — Product SCD Type 1 via Delta MERGE
Simple overwrite-in-place: matching `product_id` → update all attributes; no match → insert.
No historical rows retained (that's the defining trait of SCD1).

In [0]:
# ============================================================
# COMPLETE SCD TYPE 1 - PRODUCT DIMENSION
# SERVERLESS SAFE + DUPLICATE REPAIR
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

product_table = f"{CATALOG}.{SILVER_SCHEMA}.dim_product"

print("=" * 70)
print("PRODUCT SCD TYPE 1 PIPELINE")
print("=" * 70)
print(f"Target table: {product_table}")


# ============================================================
# STEP 1: CREATE / SEED TABLE IF NOT EXISTS
# ============================================================

if not spark.catalog.tableExists(product_table):

    seed = (
        prod_hist_clean
        .dropDuplicates(["product_id"])
        .withColumn(
            "product_sk",
            F.row_number().over(
                Window.orderBy("product_id")
            )
        )
    )

    seed.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(product_table)

    print(
        f"Seeded {product_table} with "
        f"{seed.count()} historical rows."
    )

else:

    print("Existing dim_product found. No reseeding required.")


# ============================================================
# STEP 2: LOAD EXISTING PRODUCT DIMENSION
# ============================================================

dim_product = DeltaTable.forName(
    spark,
    product_table
)

existing_df = dim_product.toDF()


# ============================================================
# STEP 3: FIND EXISTING DUPLICATE PRODUCT IDS
# ============================================================

duplicate_products = (
    existing_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_products.count()


if duplicate_count > 0:

    print(
        f"\nWARNING: Found {duplicate_count} duplicate "
        "product IDs in dim_product."
    )

    duplicate_products.show()


    # --------------------------------------------------------
    # Show the actual duplicate records
    # --------------------------------------------------------

    duplicate_ids = (
        duplicate_products
        .select("product_id")
    )

    existing_df \
        .join(
            duplicate_ids,
            "product_id",
            "inner"
        ) \
        .orderBy("product_id", "product_sk") \
        .show(truncate=False)


    # --------------------------------------------------------
    # Keep one record per product_id.
    #
    # Prefer the highest product_sk as the latest record.
    # --------------------------------------------------------

    repair_window = (
        Window
        .partitionBy("product_id")
        .orderBy(
            F.col("product_sk").desc()
        )
    )

    duplicate_rows = (
        existing_df
        .join(
            duplicate_ids,
            "product_id",
            "inner"
        )
        .withColumn(
            "_rn",
            F.row_number().over(repair_window)
        )
        .filter(
            F.col("_rn") > 1
        )
        .select("product_sk")
    )

    rows_to_delete = duplicate_rows.count()

    if rows_to_delete > 0:

        delete_ids = [
            row["product_sk"]
            for row in duplicate_rows.collect()
        ]

        delete_list = ",".join(
            [str(int(x)) for x in delete_ids]
        )

        spark.sql(f"""
            DELETE FROM {product_table}
            WHERE product_sk IN ({delete_list})
        """)

        print(
            f"Repaired {rows_to_delete} duplicate product rows."
        )

else:

    print("\nNo existing duplicate product IDs found.")


# ============================================================
# STEP 4: RELOAD AFTER REPAIR
# ============================================================

dim_product = DeltaTable.forName(
    spark,
    product_table
)

existing_df = dim_product.toDF()


# ============================================================
# STEP 5: FINAL TARGET VALIDATION BEFORE MERGE
# ============================================================

remaining_duplicates = (
    existing_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

remaining_count = remaining_duplicates.count()


if remaining_count > 0:

    remaining_duplicates.show()

    raise ValueError(
        "dim_product still contains duplicate product IDs. "
        "SCD1 MERGE stopped."
    )

print("Existing dim_product validation passed.")


# ============================================================
# STEP 6: PREPARE HISTORICAL DATA
# ============================================================

hist_products = (
    prod_hist_clean
    .withColumn(
        "_source_priority",
        F.lit(1)
    )
)


# ============================================================
# STEP 7: PREPARE INCREMENTAL DATA
# ============================================================

inc_products = (
    prod_inc_clean
    .withColumn(
        "_source_priority",
        F.lit(2)
    )
)


# ============================================================
# STEP 8: COMBINE SOURCES
# ============================================================

incoming_raw = (
    hist_products
    .unionByName(inc_products)
)


# ============================================================
# STEP 9: DEDUPLICATE INCOMING DATA
#
# Incremental data wins over historical data.
# ============================================================

product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("_source_priority").desc()
    )
)

incoming_products = (
    incoming_raw
    .withColumn(
        "_rn",
        F.row_number().over(product_window)
    )
    .filter(
        F.col("_rn") == 1
    )
    .drop(
        "_rn",
        "_source_priority"
    )
)


# ============================================================
# STEP 10: VALIDATE INCOMING DATA
# ============================================================

incoming_duplicates = (
    incoming_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

incoming_duplicate_count = incoming_duplicates.count()


if incoming_duplicate_count > 0:

    incoming_duplicates.show()

    raise ValueError(
        "Incoming product source contains duplicate product IDs."
    )


incoming_count = (
    incoming_products
    .select("product_id")
    .distinct()
    .count()
)

print(
    f"Incoming product validation passed: "
    f"{incoming_count} unique products."
)


# ============================================================
# STEP 11: GET EXISTING PRODUCT IDs
# ============================================================

existing_ids = (
    existing_df
    .select("product_id")
)


# ============================================================
# STEP 12: FIND NEW PRODUCTS
# ============================================================

new_products = (
    incoming_products
    .join(
        existing_ids,
        "product_id",
        "left_anti"
    )
)


n_new = new_products.count()


print(
    f"New products detected: {n_new}"
)


# ============================================================
# STEP 13: GENERATE PRODUCT SURROGATE KEYS
# ============================================================

max_psk = (
    existing_df
    .agg(F.max("product_sk"))
    .collect()[0][0]
)

if max_psk is None:
    max_psk = 0


new_product_window = Window.orderBy("product_id")


new_products_final = (
    new_products
    .withColumn(
        "_row_num",
        F.row_number().over(new_product_window)
    )
    .withColumn(
        "product_sk",
        F.col("_row_num") + F.lit(max_psk)
    )
    .drop("_row_num")
)


# ============================================================
# STEP 14: UPDATE EXISTING PRODUCTS
# SCD TYPE 1
# ============================================================

update_set = {
    c: f"src.`{c}`"
    for c in incoming_products.columns
    if c != "product_id"
}


# Only merge existing products for update.
existing_products_source = (
    incoming_products
    .join(
        existing_ids,
        "product_id",
        "inner"
    )
)


if existing_products_source.limit(1).count() > 0:

    (
        dim_product.alias("tgt")
        .merge(
            existing_products_source.alias("src"),
            "tgt.product_id = src.product_id"
        )
        .whenMatchedUpdate(
            set=update_set
        )
        .execute()
    )

    print(
        "Existing products updated successfully."
    )

else:

    print(
        "No existing products require updates."
    )


# ============================================================
# STEP 15: INSERT NEW PRODUCTS
# ============================================================

if n_new > 0:

    new_products_final.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(product_table)

    print(
        f"Inserted {n_new} new products."
    )

else:

    print(
        "No new products to insert."
    )


# ============================================================
# STEP 16: FINAL VALIDATION
# ============================================================

final_product_df = spark.table(product_table)


n_prod_total = final_product_df.count()

n_prod_unique = (
    final_product_df
    .select("product_id")
    .distinct()
    .count()
)

n_prod_dupes = (
    final_product_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("SCD TYPE 1 PRODUCT PIPELINE COMPLETE")
print("=" * 70)

print(
    f"Target table          : {product_table}"
)

print(
    f"Incoming products     : {incoming_count}"
)

print(
    f"New products          : {n_new}"
)

print(
    f"Total rows            : {n_prod_total}"
)

print(
    f"Unique product IDs    : {n_prod_unique}"
)

print(
    f"Duplicate product IDs : {n_prod_dupes}"
)

print("=" * 70)


if n_prod_dupes == 0:

    print(
        "SUCCESS: SCD1 validation passed."
    )

    print(
        "All product IDs are unique."
    )

else:

    print(
        "ERROR: SCD1 validation failed."
    )

    final_product_df \
        .groupBy("product_id") \
        .count() \
        .filter(F.col("count") > 1) \
        .show()

    raise ValueError(
        "Duplicate product IDs remain after SCD1 processing."
    )

PRODUCT SCD TYPE 1 PIPELINE
Target table: apex_retail1.silver_tables.dim_product
Existing dim_product found. No reseeding required.

+----------+-----+
|product_id|count|
+----------+-----+
|      1597|    2|
+----------+-----+

+----------+------------+-------------+----------------+--------------+--------------------+-------------+-------------------+------------+--------------+-------------+----------------+------------------------+-------------------+------------------+----------+------------+----------+
|product_id|product_name|product_brand|product_category|product_rating|product_review_count|product_stock|product_return_rate|product_size|product_weight|product_color|product_material|product_manufacture_date|product_expiry_date|product_shelf_life|unit_price|last_updated|product_sk|
+----------+------------+-------------+----------------+--------------+--------------------+-------------+-------------------+------------+--------------+-------------+----------------+----------------

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 4.3 — Sales Immutable Ledger via Delta MERGE + Window Dedup
Transactions never change in place (immutable ledger). We still route them through MERGE so
re-running the notebook on the same batch is idempotent (`whenNotMatchedInsertAll` only).
Strict deduplication uses a `ROW_NUMBER()` window over `transaction_id` ordered by
`transaction_date DESC`, keeping only the most recent instance of each transaction.

In [0]:
sales_table = f"{CATALOG}.{SILVER_SCHEMA}.fact_sales_ledger"

combined_sales = sales_hist_clean.unionByName(sales_inc_clean)
w = Window.partitionBy("transaction_id").orderBy(F.col("transaction_date").desc())
deduped_sales = combined_sales.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

if not spark.catalog.tableExists(sales_table):
    seed = deduped_sales.withColumn("sales_sk", F.row_number().over(Window.orderBy("transaction_id")))
    seed.write.format("delta").mode("overwrite").saveAsTable(sales_table)
    print(f"Seeded {sales_table} with {seed.count()} deduplicated rows.")
else:
    fact_sales = DeltaTable.forName(spark, sales_table)
    max_ssk = fact_sales.toDF().agg(F.max("sales_sk")).collect()[0][0] or 0
    new_txns = deduped_sales.join(fact_sales.toDF().select("transaction_id"), "transaction_id", "left_anti") \
        .withColumn("row_num", F.row_number().over(Window.orderBy("transaction_id"))) \
        .withColumn("sales_sk", F.col("row_num") + F.lit(max_ssk)).drop("row_num")
    (
        fact_sales.alias("tgt")
        .merge(new_txns.alias("src"), "tgt.transaction_id = src.transaction_id")
        .whenNotMatchedInsertAll()
        .execute()
    )

n_sales_total = spark.table(sales_table).count()
n_sales_dupes = spark.table(sales_table).groupBy("transaction_id").count().filter("count > 1").count()
print(f"Sales ledger MERGE complete. fact_sales_ledger rows = {n_sales_total}. Duplicate transaction_ids = {n_sales_dupes} (should be 0).")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Seeded apex_retail1.silver_tables.fact_sales_ledger with 2000 deduplicated rows.
Sales ledger MERGE complete. fact_sales_ledger rows = 2000. Duplicate transaction_ids = 0 (should be 0).


## 📝 MERGE Outcome — Written Explanation (Mandatory Deliverable)

**Customer (SCD Type 2).** Historical and incremental customer batches were reconciled with a
two-pass Delta `MERGE`: pass one expires the currently-active row for any `customer_id` whose
tracked attributes differ from the incoming record (`is_current = false`,
`effective_end_date = run date`); pass two inserts the new active version
(`is_current = true`, `effective_start_date = run date`) alongside brand-new customers. This
keeps full history — nothing is ever overwritten — while guaranteeing exactly one active row
per `customer_id` at any point in time, which the `Duplicate ACTIVE customer_ids` assertion
above confirms is zero.

**Product (SCD Type 1).** Because product listings don't need historical tracking, the MERGE
is a single-pass "upsert": matching `product_id`s get every column overwritten in place with
the incoming values (`whenMatchedUpdate`), and unmatched `product_id`s are appended as new
rows. This keeps `dim_product` at exactly one row per `product_id`, confirmed by the
zero-duplicate assertion above.

**Sales (Immutable Ledger).** Transactions are treated as append-only facts. Historical and
incremental batches are unioned, then a `ROW_NUMBER()` window partitioned by `transaction_id`
and ordered by `transaction_date DESC` collapses any duplicate transaction rows down to the
single latest instance before the MERGE's `whenNotMatchedInsertAll` appends only genuinely new
transactions. Re-running this notebook against the same incremental batch is therefore a
no-op for `fact_sales_ledger` — the idempotency requirement in the assignment brief.

**Why no watermarking.** The assignment explicitly prohibits watermark-based streaming
deduplication. Delta `MERGE INTO` achieves the same "don't duplicate on re-run" guarantee
through **key-based set reconciliation** instead of **time-based** state — which is both
simpler to reason about for daily batch loads and immune to late-arriving data being dropped
(a classic watermarking failure mode).

## 4.4 Surrogate Key Validation

In [0]:
for tbl, sk in [(customer_table, "customer_sk"), (product_table, "product_sk"), (sales_table, "sales_sk")]:
    df = spark.table(tbl)
    total = df.count()
    distinct_sk = df.select(sk).distinct().count()
    assert total == distinct_sk, f"{tbl}: surrogate key {sk} is NOT unique! total={total} distinct={distinct_sk}"
    print(f"✅ {tbl}: {sk} is unique across {total} rows.")

print("\nSilver layer complete:")
print(f"  {customer_table}  (SCD2)")
print(f"  {product_table}   (SCD1)")
print(f"  {sales_table}     (immutable ledger)")

✅ apex_retail1.silver_tables.dim_customer: customer_sk is unique across 2101 rows.
✅ apex_retail1.silver_tables.dim_product: product_sk is unique across 1041 rows.
✅ apex_retail1.silver_tables.fact_sales_ledger: sales_sk is unique across 2000 rows.

Silver layer complete:
  apex_retail1.silver_tables.dim_customer  (SCD2)
  apex_retail1.silver_tables.dim_product   (SCD1)
  apex_retail1.silver_tables.fact_sales_ledger     (immutable ledger)
